##### Proyecto PAPIME PE110923 DESARROLLO DE UN LABORATORIO DE ROBOTICA REMOTO PARA REALIZAR PRACTICAS DE PROGRAMACION DE ALGORITMOS DE PLANEACION Y DE NAVEGACION EN BANCOS DE PRUEBA FISICOS

# Este material fue desarrollado con el apoyo del PAPIME PE110923 de la UNAM
## Academia de Robótica

## Autores

- M.I. Erik Peña Medina
- Dr. Víctor Javier Gonzáles Villela


# Práctica 0 Nodos y Tópicos

## Objetivo

Los alumnos aprenderan las bases de la gestión información de a interacción relacionadas con el maneje de robots en entorno virtuales y en robots reales.


### Metas 

- Aprender el manejo de un espacio de trabajo de ROS 2 en Pyhton.
- EL manejo de herramientas básicas de programación para la robótica.
- Los alummnos programarán una arquitectura de Nodo Publicador, Nodo Subcriptor y Nodo Subcriptor Publicador.
  

### Contribución al perfil del egresado

La siguiente práctica contribuye en los siguientes puntos al perfil del egresado:

#### Aptitudes y habilidades

- Para modelar, simular e interpretar el comportamiento de los sistemas mecatrónicos.
- Para desarrollar, operar y mantener procesos productivos que impliquen la transformación de materia, energía e información.
- Para diseñar, construir, operar y mantener los sistemas mecatrónicos y sus componentes.

#### Actitudes

- Ser creativo e innovador.
- Tener confianza en su preparación académica.
- Comprometido con su actualización, superación y competencia profesional.

#### De tipo social

- Promover el cambio en la mentalidad frente a la competitividad internacional.



## Rúbrica de evaluación

La evaluación de la práctica contará de los siguientes puntos:

| Elemento | Porcentaje |
| ------:| ----------- |
| ***Previo*** | 25% | 
| **Desarrollo** | 25% |
| **Resultado**  | 25% |
| **Conclusiones** | 25% |


Los siguientes elementos se evaluaran con base en los siguientes criterios:

| Elemento | Malo | Regular | Bueno | Muy bueno| 
| ------:| ------ | --------| ------| --------  |
| ***Previo*** | El previo no contiene los elemtos solocitados (0%)| El previo contiene parcialmente la infomración solocitada (10%) |  El previo contiene la infomración solocitada sin referencias (20%) | El previo contiene la información solicitada con referencias (25%) |
| **Desarrollo** | El desarrollo de la práctica no fue realizada (0%) | El desarrollo de la práctica fue pacialmente realizado (10%) | La práctica fue realizada pero no fue realizada correctamente (20%) | La práctica fue realizada en su totalidad correctamente (25%) |
| **Resultado**  | No se entregarón los resultados de la práctica (0%)| Se entregaron pracialmente los resultados (10%) | Se entregaron los resultados sin descripción (20%) | Se entregaron los resultados de la práctica y son interpetrables(25%) |
| **Conclusiones** | No se entragaron conclusiones (0%) | Las conclusciones no están relacionadas con el objetivo de la práctica (5%) | Las conclusciones se relacionan con el objetivo de práctica (10%) | Las concluscions se relacionana con el objetivo de la práctica y se basan en los resultados obtenidos (25%) | 



## Previo de la Práctica 0

Se le solicita al alumno investigar y colocar la sigueinte información en esta sección:

- La estructura de paquetes en ROS 2:

    Es la unidad básica de organización del Software. Contiene lo necesario para que un programa pueda ejecutarse incluyendo el código fuente, archivos de configuración y dependencias. Entre los archivos principales podemos encontrar package.xml, el cual define la información del paquete, y setup.py, el cual permite su instalación y ejecución

Dentro de este paquete existe un directorio que lleva el mismo nombre, este contiene los scripts de los nodos y un archivo init.py que nos permite su reconocimiento como módulo de python. Esa estructura facilita la organización, reutilización y escalabilidad de los sistemas robóticos 
   
- Definición de Nodos de Tópicos:

    Un nudo en ROS 2 es un proceso que realiza una tarea específica dentro de un sistema. Los nodos pueden comunicarse entre sí mediante los tópicos, utilizando el modelo publicador-suscriptor.

Un tópico es un canal de comunicación en el cual un nodo publicador puede enviar información, mientras que uno o varios  nodos suscriptores reciben esos datos. Los mensajes transmitidos tienen un tipo de dato definido, como float 64 en nuestro caso, lo que garantiza la correcta interpretación de la información.

## Desarrollo de la práctica 
En el desarrollo de esta práctica creamos un paquete en ROS 2 utilizando python, en el cual implementamos dos nodos que se comunican mediante tópicos.

El primer nudo corresponde a un publicador que se encarga de generar una señal senoidal que representa valores de velocidad en revoluciones por minuto. Esta señal se definió con un Rango de 0 a 1 utilizando una función seno ajustada, y se publica periódicamente mediante un temporizador en el tópico /rpm_ropic.

Nodo publicador:

import rclpy
from rclpy.node import Node
from std_msgs.msg import Float64
import math

class RPMPublisher(Node):
    def __init__(self):
        super().__init__("rpm_publisher")

        self.t = 0.0

        self.publisher_ = self.create_publisher(
            Float64,
            'rpm_topic',
            10
        )
        
        self.timer = self.create_timer(0.1, self.publish_signal)
        
        self.get_logger().info('Nodo publicador activo')
    
    def publish_signal(self):
        msg = Float64()

        # Señal senoidal en rango [0,1]
        rpm = 0.5 * (math.sin(self.t) + 1)
        msg.data = rpm

        self.publisher_.publish(msg)

        self.get_logger().info(f'RPM: {rpm:.3f}')

        self.t += 0.1


def main(args=None):
    rclpy.init(args=args)
    node = RPMPublisher()
    rclpy.spin(node)
    rclpy.shutdown()

if __name__ == '__main__':
    main()

Después implementamos un nuevo suscriptor-publicador, el cual recibe los datos del tópico /rpm_topic, realiza la conversión de unidades de rpm a radianes por segundo y publica el resultado en el tópico /rad_s_topic.

Nodo suscriptor-publicador:

import rclpy
from rclpy.node import Node
from std_msgs.msg import Float64
import math

class Converter(Node):
    def __init__(self):
        super().__init__('converter_node')

        self.create_subscription(
            Float64,
            'rpm_topic',
            self.callback,
            10
        )
        
        self.publisher_ = self.create_publisher(
            Float64,
            'rad_s_topic',
            10
        )
        
        self.get_logger().info('Nodo convertidor activo')

    def callback(self, msg):
        rpm = msg.data

        # Conversión a rad/s
        rad_s = (2 * math.pi / 60) * rpm

        new_msg = Float64()
        new_msg.data = rad_s

        self.publisher_.publish(new_msg)

        self.get_logger().info(f"RPM: {rpm:.3f} -> RAD/S: {rad_s:.3f}")


def main(args=None):
    rclpy.init(args=args)
    node = Converter()
    rclpy.spin(node)
    rclpy.shutdown()

if __name__ == '__main__':
    main()

Finalmente, compilamos el paquete utilizando colcon build y ejecutamos los nodos en distintas terminales. Para verificar el funcionamiento, se utilizó el comandorRos2 topic echo, el cual permitió observar los valores publicados en los tópicos, confirmando la correcta comunicación y transformación de nuestra señal.

## Resultados

Logramos establecer correctamente la comunicación entre los nodos mediante el uso de tópicos en ROS 2. El nodo publicador generó la señal senoidal en el rango esperado y la transmitió de forma continua.

El nodo suscriptor-publicador recibió esa señal sin errores, realizó la conversión de unidades de manera adecuada y publicó los nuevos valores en un segundo tópico. La información pudo ser visualizada en tiempo real mediante herramientas de ROD 2, lo cual nos permitió comprobar el correcto funcionamiento del sistema.

 En general, el sistema operó de una forma estable, lo cual nos permitió cumplir con los objetivos planteados en la práctica y demostrando el uso adecuado del modelo de comunicación basado en tópicos.


### Vídeos

A continuación se deja un video para verificar el correcto funcionamiento del sistema.

[![Ver video](https://img.youtube.com/vi/u93Y8kLrXFE/0.jpg)](https://www.youtube.com/watch?v=u93Y8kLrXFE)


    Ramiro Sánchez Leonardo